In [2]:
import pandas as pd

my_books = pd.read_csv('../data/liked_books_full.csv')

In [3]:
my_books['book_id'] = my_books['book_id'].astype(str)
my_books = my_books[['book_id', 'title', 'user_id', 'average_rating']]
my_books = my_books.rename(columns={'average_rating':'rating'})
my_books.head()


,book_id,title,user_id,rating
0,49455,Notes from Underground,-1,4.17
1,1303,The 48 Laws of Power,-1,4.10
2,41881472,The Psychology of Money,-1,4.29
3,4865,How to Win Friends & Influence People,-1,4.22
4,43808723,Everything Is F*cked: A Book About Hope,-1,3.67


In [4]:
csv_book_mapping = {}

with open("../data/book_id_map.csv", 'r') as f:
    while True:
        line = f.readline()
        if not line:
            break
        csv_id, book_id = line.strip().split(',')
        csv_book_mapping[csv_id] = book_id

In [5]:
book_set = set(my_books['book_id'])

In [6]:
overlap_users = {}

with open("../data/filtered_interactions.csv", 'r') as f:
    while True:
        line = f.readline()
        if not line:
            break
        user_id, csv_id, _, rating, _ = line.strip().split(',')
        book_id = csv_book_mapping.get(csv_id)
        if book_id in book_set:
            if user_id not in overlap_users:
                overlap_users[user_id] = 1
            else:
                overlap_users[user_id] += 1

In [7]:
len(overlap_users)

465199

In [8]:
filter_overlap_users = set([k for k in overlap_users if overlap_users[k] > my_books.shape[0]/5])

In [9]:
len(filter_overlap_users)

83236

In [10]:
# interactions_list = []

# with open('../data/goodreads_interactions.csv', 'r') as f:
#     while True:
#         line = f.readline()
#         if not line:
#             break
#         user_id, csv_id, _, rating, _ = line.strip().split(',')
#         if user_id in filter_overlap_users:
#             book_id = csv_book_mapping.get(csv_id)
#             interactions_list.append([user_id, book_id, rating])

In [11]:
# interactions = pd.DataFrame(interactions_list, columns=['user_id', 'book_id', 'rating'])
# len(interactions_list)

In [12]:
chunks = []
chunk_data = []
chunk_size = 10000

with open('../data/filtered_interactions.csv', 'r') as f:
    next(f)  # Skip header
    
    for line in f:
        try:
            parts = line.strip().split(',')
            if len(parts) >= 5:
                user_id, csv_id, _, rating, _ = parts[:5]
                
                if user_id in filter_overlap_users:
                    book_id = csv_book_mapping.get(csv_id)
                    if book_id is not None:
                        chunk_data.append([int(user_id), int(book_id), float(rating)])
                        
                        if len(chunk_data) >= chunk_size:
                            chunk_df = pd.DataFrame(chunk_data, columns=['user_id', 'book_id', 'rating'])
                            chunks.append(chunk_df)
                            chunk_data = []
                            print(f"Created chunk {len(chunks)}")
                            
        except (ValueError, IndexError):
            continue
    
    # Handle remaining data
    if chunk_data:
        chunk_df = pd.DataFrame(chunk_data, columns=['user_id', 'book_id', 'rating'])
        chunks.append(chunk_df)

# Concat all chunks at once (more efficient)
interactions = pd.concat(chunks, ignore_index=True)

Created chunk 1
Created chunk 2
Created chunk 3
Created chunk 4
Created chunk 5
Created chunk 6
Created chunk 7
Created chunk 8
Created chunk 9
Created chunk 10
Created chunk 11
Created chunk 12
Created chunk 13
Created chunk 14
Created chunk 15
Created chunk 16
Created chunk 17
Created chunk 18
Created chunk 19
Created chunk 20
Created chunk 21
Created chunk 22
Created chunk 23
Created chunk 24
Created chunk 25
Created chunk 26
Created chunk 27
Created chunk 28
Created chunk 29
Created chunk 30
Created chunk 31
Created chunk 32
Created chunk 33
Created chunk 34
Created chunk 35
Created chunk 36
Created chunk 37
Created chunk 38
Created chunk 39
Created chunk 40
Created chunk 41
Created chunk 42
Created chunk 43
Created chunk 44
Created chunk 45
Created chunk 46
Created chunk 47
Created chunk 48
Created chunk 49
Created chunk 50
Created chunk 51
Created chunk 52
Created chunk 53
Created chunk 54
Created chunk 55
Created chunk 56
Created chunk 57
Created chunk 58
Created chunk 59
Create

In [13]:
interactions.head()

,user_id,book_id,rating
0,8,1885,4.0
1,8,2657,4.0
2,8,10210,5.0
3,8,4671,2.0
4,8,156538,4.0


In [14]:
len(interactions)

38238088

In [15]:
def concat_in_chunks(large_df, small_df, chunk_size=1000000):
    # Start with the small dataframe
    result = small_df.copy()
    
    # Add large dataframe in chunks
    for i in range(0, len(large_df), chunk_size):
        chunk = large_df.iloc[i:i+chunk_size]
        result = pd.concat([result, chunk], ignore_index=True)
        
        # Force garbage collection
        import gc
        gc.collect()
        
        print(f"Processed {min(i+chunk_size, len(large_df)):,} / {len(large_df):,} rows")
    
    return result

# Use it
my_books_subset = my_books[['user_id', 'book_id', 'rating']].copy()
interactions = concat_in_chunks(interactions, my_books_subset)

Processed 1,000,000 / 38,238,088 rows
Processed 2,000,000 / 38,238,088 rows
Processed 3,000,000 / 38,238,088 rows
Processed 4,000,000 / 38,238,088 rows
Processed 5,000,000 / 38,238,088 rows
Processed 6,000,000 / 38,238,088 rows
Processed 7,000,000 / 38,238,088 rows
Processed 8,000,000 / 38,238,088 rows
Processed 9,000,000 / 38,238,088 rows
Processed 10,000,000 / 38,238,088 rows
Processed 11,000,000 / 38,238,088 rows
Processed 12,000,000 / 38,238,088 rows
Processed 13,000,000 / 38,238,088 rows
Processed 14,000,000 / 38,238,088 rows
Processed 15,000,000 / 38,238,088 rows
Processed 16,000,000 / 38,238,088 rows
Processed 17,000,000 / 38,238,088 rows
Processed 18,000,000 / 38,238,088 rows
Processed 19,000,000 / 38,238,088 rows
Processed 20,000,000 / 38,238,088 rows
Processed 21,000,000 / 38,238,088 rows
Processed 22,000,000 / 38,238,088 rows
Processed 23,000,000 / 38,238,088 rows
Processed 24,000,000 / 38,238,088 rows
Processed 25,000,000 / 38,238,088 rows
Processed 26,000,000 / 38,238,088 

In [16]:
interactions['book_id'] = interactions['book_id'].astype(str)
interactions['user_id'] = interactions['user_id'].astype(str)
interactions['rating'] = pd.to_numeric(interactions['rating'])

In [17]:
interactions['user_index'] = interactions['user_id'].astype('category').cat.codes

In [18]:
len(interactions['user_index'].unique())

83237

In [19]:
interactions['book_index'] = interactions['book_id'].astype('category').cat.codes

In [20]:
len(interactions['book_index'].unique())

62337

In [21]:
from scipy.sparse import coo_matrix

rating_mat_coo = coo_matrix((interactions['rating'], (interactions['user_index'], interactions['book_index'])))

In [22]:
rating_mat_coo

<COOrdinate sparse matrix of dtype 'float64'
	with 38238114 stored elements and shape (83237, 62337)>

In [23]:
ratings_mat = rating_mat_coo.tocsr()

In [24]:
# interactions[interactions['user_id']=='-1']
interactions.head(-5)

,user_id,book_id,rating,user_index,book_index
0,-1,49455,4.17,0,46640
1,-1,1303,4.10,0,6727
2,-1,41881472,4.29,0,44655
3,-1,4865,4.22,0,46407
4,-1,43808723,3.67,0,45183
...,...,...,...,...,...
38238104,876101,32261,0.00,80850,40130
38238105,876101,50798,3.00,80850,46974
38238106,876101,11245,5.00,80850,2708
38238107,876101,236709,0.00,80850,29532


In [25]:
my_index = 0

In [26]:
from sklearn.metrics.pairwise import cosine_similarity

similarity = cosine_similarity(ratings_mat[my_index,:], ratings_mat).flatten()

In [27]:
similarity[2]

np.float64(0.028619400314882772)

In [28]:
import numpy as np
indices = np.argpartition(similarity, -15)[-15:]

In [29]:
indices

array([ 2064, 78524, 78012, 79225, 74020, 73853, 74241, 67091, 78308,
       66453, 74156, 75114, 79678, 78186,     0])

In [30]:
similar_users = interactions[interactions['user_index'].isin(indices)].copy()

In [31]:
similar_users = similar_users[similar_users['user_id'] != '-1']

In [32]:
similar_users

,user_id,book_id,rating,user_index,book_index
9646895,111237,48855,0.0,2064,46447
9646896,111237,10884,0.0,2064,1986
9646897,111237,960,4.0,2064,61283
9646898,111237,2429135,4.0,2064,30219
9646899,111237,7766027,0.0,2064,56133
...,...,...,...,...,...
38198820,843232,485894,3.0,79678,46398
38198821,843232,153747,5.0,79678,10856
38198822,843232,1618,5.0,79678,14000
38198823,843232,18254,5.0,79678,19393


In [44]:
book_recs = similar_users.groupby('book_id').rating.agg(['count', 'mean'])

In [45]:
book_recs

,count,mean
book_id,,
10037,1,0.0
1005,1,0.0
10098912,1,4.0
10210,6,2.0
10357575,1,0.0
...,...,...
97411,1,0.0
976,1,3.0
9864913,1,5.0


In [1]:
book_titles = pd.read_json("../data/books_titles.json")
book_titles['book_id'] = book_titles['book_id'].astype(str)

NameError: name 'pd' is not defined

In [36]:
book_recs = book_recs.merge(book_titles, how='inner', on='book_id')

In [43]:
book_recs

,book_id,count,mean,title,ratings,url,cover_image,mod_title,adjusted_count,score
89,18405,3,4.333333,Gone with the Wind,887935,https://www.goodreads.com/book/show/18405.Gone...,https://images.gr-assets.com/books/1328025229m...,gone with the wind,0.000010,0.000044
201,51496,3,4.333333,The Strange Case of Dr. Jekyll and Mr. Hyde,229898,https://www.goodreads.com/book/show/51496.The_...,https://images.gr-assets.com/books/1318116526m...,the strange case of dr jekyll and mr hyde,0.000039,0.000170


In [ ]:
book_recs['adjusted_count'] = book_recs['count'] * (book_recs['count']/book_recs['ratings'])
book_recs['score'] = book_recs['mean']*book_recs['adjusted_count']
book_recs = book_recs[~book_recs['book_id'].isin(my_books['book_id'])]
my_books['mod_title'] = my_books['title'].str.replace("[^a-zA-Z0-9 ]", "", regex=True)
my_books['mod_title'] = my_books['mod_title'].str.replace("\s+", " ", regex=True)
book_recs = book_recs[~book_recs['mod_title'].isin(my_books['mod_title'])]
book_recs = book_recs[book_recs['count']>2]
book_recs = book_recs[book_recs['mean']>4]
top_recs = book_recs.sort_values('score', ascending=False)

<>:5: SyntaxWarning: invalid escape sequence '\s'
<>:5: SyntaxWarning: invalid escape sequence '\s'
C:\Users\Harshvardhan\AppData\Local\Temp\ipykernel_22100\3675059281.py:5: SyntaxWarning: invalid escape sequence '\s'
  my_books['mod_title'] = my_books['mod_title'].str.replace("\s+", " ", regex=True)


In [42]:
top_recs

,book_id,count,mean,title,ratings,url,cover_image,mod_title,adjusted_count,score
201,51496,3,4.333333,The Strange Case of Dr. Jekyll and Mr. Hyde,229898,https://www.goodreads.com/book/show/51496.The_...,https://images.gr-assets.com/books/1318116526m...,the strange case of dr jekyll and mr hyde,0.000039,0.000170
89,18405,3,4.333333,Gone with the Wind,887935,https://www.goodreads.com/book/show/18405.Gone...,https://images.gr-assets.com/books/1328025229m...,gone with the wind,0.000010,0.000044
